# Lab 3.1: Data Download from Google Earth Engine

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This notebook demonstrates how to acquire and download Sentinel-2 satellite imagery and CORINE land cover maps using Google Earth Engine (GEE) for land classification tasks.

---

## Project Milestone Overview

| Lab | Milestone | Status |
|-----|-----------|--------|
| Lab 1 | HPC Access Setup | ✅ Previous |
| Lab 2 | Jupyter-JSC & Git | ✅ Previous |
| Lab 3.1 | **Data Download (GEE)** | 🔄 **Current** |
| Lab 3.2 | Data Preprocessing | ⬜ Next |
| Lab 4 | Understanding Transformers | ⬜ Next |
| Lab 4.1 | Training on Sentinel-2 Data | ⬜ Next |
| Lab 5 | Distributed Training (Multi-GPU) | ⬜ Next |
| Lab 6 | Validation & Performance Metrics | ⬜ Next |
| Lab 7 | Foundation Models & TerraToRCH | ⬜ Final |

---

## What You'll Learn

By the end of this lab, you will:
- Authenticate and initialize Google Earth Engine (GEE)
- Query Sentinel-2 Level 2A imagery collections
- Filter by date, location, and cloud cover
- Load CORINE land cover labels
- Export S2 and CORINE as GeoTIFF files
- Work with geospatial data in GEE and geemap

## Quick Start

This lab focuses on the **Data Acquisition** phase of the project pipeline:

```
Data Download from GEE (Lab 3.1) ← You are here
    ↓
Data Preprocessing (Lab 3.2)
    ↓
CORINE Label Extraction (Lab 3.2)
    ↓
Training Data Preparation (Lab 4)
    ↓
Model Training (Lab 4.1, 5)
    ↓
Validation & Evaluation (Lab 6)
    ↓
Foundation Models (Lab 7)
```

---

## Overview

This lab focuses on:
1. **GEE Authentication**: Setting up credentials
2. **Querying Sentinel-2 Data**: Finding imagery by date, location, and quality
3. **CORINE Label Lookup**: Accessing land cover classification data
4. **Export to GeoTIFF**: Downloading data for local processing
5. **Validation**: Verifying data quality before preprocessing

## Part 1: Setup and Authentication

First, authenticate with Google Earth Engine and initialize the API.

In [2]:
import ee
import os
import geemap
import numpy as np
from datetime import datetime
import json

Matplotlib created a temporary cache directory at /tmp/matplotlib-di1v752o because the default path (/p/home/jusers/hashim1/jureca/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


### Authenticate with Google Earth Engine

Run the following cell. It will open a browser window for authentication.

In [3]:
# Authenticate with GEE (first time only)
ee.Authenticate()

Enter verification code:  4/1ASc3gC2YpoGNIBesimhNDYl5218Pyewbrds_HUoe6aIZN-za-sJRCpNzpfQ



Successfully saved authorization token.


### Initialize Earth Engine

In [4]:
# Initialize the Earth Engine API
ee.Initialize()

---

## Part 2: Define Region of Interest (ROI)

Specify your study area as a bounding box or polygon.

### Example: Define ROI from Coordinates

In [5]:
# Define your Region of Interest (ROI) as a bounding box
# Example: Iceland region (modify for your area)
# Format: [min_lon, min_lat, max_lon, max_lat]

roi_coords = [-21.0, 63.5, -13.0, 66.5]  # Iceland example
roi = ee.Geometry.Rectangle(roi_coords)

print(f"ROI defined: {roi_coords}")
print(f"ROI area: {roi.area().getInfo()} m²")

ROI defined: [-21.0, 63.5, -13.0, 66.5]
ROI area: 125097167114.58807 m²


### Visualize ROI

In [6]:
# Create map and visualize ROI
Map = geemap.Map(center=[64.5, -17.0], zoom=6)
Map.addLayer(roi, {'color': 'FF0000'}, 'Region of Interest')
Map

Map(center=[64.5, -17.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

---

## Part 3: Query Sentinel-2 Data

Find and filter Sentinel-2 Level 2A imagery for your ROI.

### Query Sentinel-2 Collection

In [7]:
# Define date range (modify as needed)
start_date = '2018-05-01'
end_date = '2018-05-31'

# Query Sentinel-2 L2A collection
s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))  # Less than 10% cloud cover

# Get collection info
num_images = s2_collection.size().getInfo()
print(f"Found {num_images} Sentinel-2 images with <10% cloud cover")

# List all available images
image_list = s2_collection.toList(num_images)
for i in range(min(num_images, 5)):  # Show first 5
    img = ee.Image(image_list.get(i))
    date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    cloud = img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
    print(f"  {i+1}. {date} - Cloud: {cloud}%")

Found 25 Sentinel-2 images with <10% cloud cover
  1. 2018-05-01 - Cloud: 0.941238%
  2. 2018-05-01 - Cloud: 7.777803%
  3. 2018-05-01 - Cloud: 8.055282%
  4. 2018-05-01 - Cloud: 9.505635%
  5. 2018-05-01 - Cloud: 7.821929%


### Select Single Image

In [8]:
# Select the first (best) image
s2_image = s2_collection.first()

# Get image metadata
date = ee.Date(s2_image.get('system:time_start')).format('YYYY-MM-dd').getInfo()
cloud = s2_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
tile_id = s2_image.get('MGRS_TILE').getInfo()

print(f"Selected image:")
print(f"  Date: {date}")
print(f"  Cloud cover: {cloud}%")
print(f"  MGRS Tile: {tile_id}")

Selected image:
  Date: 2018-05-01
  Cloud cover: 0.941238%
  MGRS Tile: 28VDR


### Visualize Sentinel-2 Image

In [9]:
# Define visualization parameters for RGB composite
vis_params = {
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue
    'min': 0,
    'max': 3000
}

# Create map and add layers
Map = geemap.Map()
Map.centerObject(roi, 8)
Map.addLayer(s2_image, vis_params, 'Sentinel-2 RGB')
Map.addLayer(roi, {'color': 'FF0000'}, 'ROI Boundary')
Map

Map(center=[65.02538648535148, -17.00000000000003], controls=(WidgetControl(options=['position', 'transparent_…

---

## Part 4: Access CORINE Land Cover Data

Load CORINE classification labels for the same region.

### Load CORINE Collection

In [10]:
# Load CORINE land cover data
# Use the version that covers your time period
corine = ee.ImageCollection('COPERNICUS/CORINE/V20/100m') \
    .filterBounds(roi) \
    .filterDate('2017-01-01', '2019-01-01') \
    .first()

# Get first band (classification)
corine_classification = corine.select('classification')

print("CORINE land cover data loaded")
print(f"Projection: {corine.projection().getInfo()}")

CORINE land cover data loaded
Projection: {'type': 'Projection', 'crs': 'EPSG:3035', 'transform': [100, 0, 900000, 0, -100, 5500000]}


### CORINE Class Mapping

Here are the main CORINE land cover classes (12 classes used in the project):

In [11]:
# CORINE class mapping (12 main classes)
corine_classes = {
    1: 'Urban fabric',
    2: 'Industrial/commercial',
    3: 'Arable land',
    4: 'Permanent crops',
    5: 'Pastures',
    6: 'Complex agriculture',
    7: 'Herbaceous vegetation',
    8: 'Forests',
    9: 'Herbaceous/woody vegetation',
    10: 'Water',
    11: 'Clouds/shadows',
    12: 'No data'
}

print("CORINE Land Cover Classes:")
for class_id, class_name in corine_classes.items():
    print(f"  {class_id:2d}: {class_name}")

CORINE Land Cover Classes:
   1: Urban fabric
   2: Industrial/commercial
   3: Arable land
   4: Permanent crops
   5: Pastures
   6: Complex agriculture
   7: Herbaceous vegetation
   8: Forests
   9: Herbaceous/woody vegetation
  10: Water
  11: Clouds/shadows
  12: No data


### Visualize CORINE

In [12]:
# Create visualization with color palette
corine_vis = {
    'min': 1,
    'max': 12,
    'palette': [
        'FF0000',  # 1: Urban - Red
        'FF00FF',  # 2: Industrial - Magenta
        'FFFF00',  # 3: Arable - Yellow
        'FF9900',  # 4: Crops - Orange
        'CCFF00',  # 5: Pastures - Yellow-green
        'FFCC00',  # 6: Complex - Gold
        '00FF00',  # 7: Herbaceous - Green
        '006600',  # 8: Forests - Dark green
        '00FF99',  # 9: Mixed veg - Light green
        '0000FF',  # 10: Water - Blue
        '00FFFF',  # 11: Clouds - Cyan
        '999999'   # 12: No data - Gray
    ]
}

Map = geemap.Map()
Map.centerObject(roi, 8)
Map.addLayer(corine_classification, corine_vis, 'CORINE Classification')
Map.addLayer(roi, {'color': '000000'}, 'ROI Boundary')
Map

EEException: Image.select: Band pattern 'classification' did not match any bands. Available bands: [landcover]

---

## Part 5: Export Data as GeoTIFF

Export Sentinel-2 and CORINE data to your Google Drive or storage system.

### Export Sentinel-2 Bands (Individual)

Export key Sentinel-2 bands at 10m resolution.

In [13]:
# Select 11 standard Sentinel-2 bands for classification
s2_bands = s2_image.select(['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'SCL'])

# Resample to 10m (reproject all to match 10m resolution)
s2_10m = s2_bands.resample('bilinear').reproject(
    crs=s2_image.select('B2').projection(),
    scale=10
)

# Export task
export_task = ee.batch.Export.image.toDrive(
    image=s2_10m,
    description='s2_sentinel2_10bands_10m',
    folder='GEE_Exports',
    fileNamePrefix='s2_sentinel2_10m',
    scale=10,
    region=roi,
    fileFormat='GeoTIFF',
    maxPixels=1e10
)

print("Starting export task: Sentinel-2 10-band image")
export_task.start()
print(f"Task: {export_task.id}")

Starting export task: Sentinel-2 10-band image
Task: ALALYTJA5YBERQQFUHV273A4


### Export CORINE Classification

In [14]:
# Resample CORINE to 10m to match Sentinel-2
corine_10m = corine_classification.resample('mode').reproject(
    crs=s2_image.select('B2').projection(),
    scale=10
)

# Export task
export_corine = ee.batch.Export.image.toDrive(
    image=corine_10m,
    description='corine_lulc_10m',
    folder='GEE_Exports',
    fileNamePrefix='corine_lulc_10m',
    scale=10,
    region=roi,
    fileFormat='GeoTIFF',
    maxPixels=1e10
)

print("Starting export task: CORINE land cover")
export_corine.start()
print(f"Task: {export_corine.id}")

Starting export task: CORINE land cover
Task: NWDMDBHNLRDU42DIUFZMDZC3


### Monitor Export Tasks

In [15]:
# Check status of all tasks
tasks = ee.batch.Task.list()

print("Current export tasks:")
for task in tasks:
    print(f"  {task.config['description']}: {task.status()['state']}")

Current export tasks:
  corine_lulc_10m: READY
  s2_sentinel2_10bands_10m: FAILED
  S2_Tile_4_20240913: COMPLETED
  S2_Tile_3_20240629: COMPLETED
  S2_Tile_2_20240913: COMPLETED
  S2_Tile_1_20240923: COMPLETED
  S2_Tile_4_20240707: COMPLETED
  S2_Tile_3_20240915: COMPLETED
  S2_Tile_2_20240913: COMPLETED
  S2_Tile_1_20240629: COMPLETED
  S2_Tile_4_20240629: COMPLETED
  S2_Tile_3_20240610: COMPLETED
  S2_Tile_2_20240915: COMPLETED
  S2_Tile_1_20240629: COMPLETED
  S2_Tile_4_20240629: FAILED
  S2_Tile_3_20240610: FAILED
  S2_Tile_2_20240915: FAILED
  S2_Tile_1_20240629: FAILED
  sentinel2_iceland_20240707: COMPLETED
  sentinel2_iceland_20240915: COMPLETED
  sentinel2_iceland_20240913: COMPLETED
  sentinel2_iceland_20240629: COMPLETED
  sentinel2_iceland_20240923: COMPLETED
  sentinel2_iceland_20240610: COMPLETED
  sentinel2_iceland_20240826: COMPLETED
  sentinel2_iceland_20240629: COMPLETED
  sentinel2_iceland_20240610: COMPLETED
  sentinel2_iceland_20240826: COMPLETED
  sentinel2_icelan

---

## Part 6: Alternative - Geemap Export

Use geemap's simplified export functions (alternative method).

In [21]:
# Alternative: Use geemap to export directly
# This exports to your local drive/HPC storage

# Example: Export S2 image
output_path = '/p/scratch/training2600/hashim1/s2_sentinel2_10m.tif'
geemap.ee_export_image(s2_10m, filename=output_path, scale=10, region=roi, file_per_band=False)

# Example: Export CORINE
# output_path = '/path/to/hpc/storage/corine_lulc_10m.tif'
# geemap.ee_export_image(corine_10m, filename=output_path, scale=10, region=roi, file_per_band=False)

print("Uncomment and modify the paths above for direct local export")

Generating URL ...
An error occurred while downloading.
Total request size (110102720 bytes) must be less than or equal to 50331648 bytes.
Uncomment and modify the paths above for direct local export


---

## Part 7: Data Quality Validation

Verify that downloaded data is correct before proceeding to preprocessing.

### Get Export Metadata

In [ ]:
# Get S2 image properties
s2_props = s2_image.getInfo()['properties']

print("Sentinel-2 Image Properties:")
print(f"  MGRS Tile: {s2_props.get('MGRS_TILE')}")
print(f"  Date: {s2_props.get('SENSING_TIME')}")
print(f"  Cloud Cover: {s2_props.get('CLOUDY_PIXEL_PERCENTAGE')}%")
print(f"  Solar Zenith: {s2_props.get('MEAN_SOLAR_ZENITH_ANGLE')}°")

### Verify Data Dimensions

In [ ]:
# Check dimensions of exported data
s2_10m_info = s2_10m.getInfo()

print("Exported Sentinel-2 data dimensions:")
print(f"  Bands: {len(s2_10m_info['properties'].get('band_names', []))}")
print(f"  Projection: {s2_10m_info['properties'].get('PROJECTION')}")
print(f"  Expected dimensions: ~10,980 x 10,980 pixels at 10m resolution")

---

## Summary

This notebook demonstrated how to:

1. **Authenticate GEE**: Connect to Google Earth Engine API
2. **Query S2 data**: Find cloud-free Sentinel-2 imagery for your ROI
3. **Access CORINE**: Load land cover classification labels
4. **Export as GeoTIFF**: Download both datasets for local processing
5. **Validate data**: Check properties and dimensions

The exported GeoTIFF files are ready for preprocessing in **Lab 3.2**.

---

## What's Next?

### Before Moving to Lab 3.2

**1. Verify Export Completion**
   - Check Google Drive or HPC storage for downloaded files
   - Confirm both S2 and CORINE GeoTIFFs are present
   - Verify file sizes (~500 MB - 1 GB each)

**2. Update Paths**
   - Note the full paths to your downloaded files
   - Update paths in Lab 3.2 preprocessing scripts

**3. Data Inspection**
   - Open GeoTIFFs in QGIS to visually verify
   - Check that data covers your ROI correctly
   - Verify no data or artifacts

### Next Lab: Lab 3.2 - Data Preprocessing

In **Lab 3.2**, you'll:
- Read the downloaded GeoTIFF files with GDAL
- Extract patches for training
- Align Sentinel-2 and CORINE data
- Create training/validation/test splits
- Prepare data for Lab 4 (model training)

### Data Pipeline Recap

```python
# Lab 3.1 produces:
Sentinel-2 GeoTIFF (downloaded from GEE)
CORINE GeoTIFF (downloaded from GEE)
    ↓
# Lab 3.2 processes:
Read GeoTIFFs with GDAL
Extract 3x3 or 5x5 patches
Create training/validation datasets
    ↓
# Lab 4 uses:
Training patches with labels
```

---

## Resources & References

- **Google Earth Engine API**: https://developers.google.com/earth-engine
- **Geemap Documentation**: https://geemap.org/
- **Sentinel-2 Product Specification**: https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2
- **CORINE Land Cover**: https://www.eea.europa.eu/publications/COR0-landcover
- **GEE Data Catalog**: https://developers.google.com/earth-engine/datasets

---

## Troubleshooting & FAQ

**Q: GEE authentication fails - what should I do?**
- First-time users need a Google Cloud project with Earth Engine API enabled
- Visit https://earthengine.google.com/ to sign up for free access
- Wait for approval (usually 24-48 hours)

**Q: How long does export take?**
- Small ROIs (<10,000 km²): 10-30 minutes
- Large ROIs: Can take hours
- Check status in GEE Code Editor Tasks tab

**Q: Can I export directly to HPC storage?**
- GEE can only export to Google Drive or Cloud Storage
- Download from Drive to HPC via rclone or wget
- Alternatively, use geemap's local export option

**Q: My CORINE and S2 don't align - why?**
- Different original resolutions (CORINE 100m, S2 10m)
- Lab 3.2 handles resampling and alignment
- Ensure both are reprojected to same CRS

---

**Course Contact**: Refer to course materials for instructor email and office hours  
**Last Updated**: January 2026